In [1]:
from biovec.models import prot_vec
import numpy as np    
import pandas as pd



D:\Programas\Anaconda3\envs\prospeccion\lib\site-packages\gensim\utils.py:1197: UserWarning: detected Windows; aliasing chunkize to chunkize_serial
  warnings.warn("detected Windows; aliasing chunkize to chunkize_serial")


In [3]:
pv2 = prot_vec.load_protvec('swissprot-reviewed-protvec.model')
res = pv2.to_vecs("ATEEE")


D:\Programas\Anaconda3\envs\prospeccion\lib\site-packages\smart_open\smart_open_lib.py:398: UserWarning: This function is deprecated, use smart_open.open instead. See the migration notes for details: https://github.com/RaRe-Technologies/smart_open/blob/master/README.rst#migrating-to-the-new-open-function
  'See the migration notes for details: %s' % _MIGRATION_NOTES_URL


In [4]:
datos_pos = pd.read_csv('positivos.csv',  index_col = 0)
datos_neg =pd.read_csv('negativos.csv',  index_col = 0)
df = pd.concat([datos_pos,datos_neg])
df.reset_index(drop=True);
df.tail()

,sequence,length,molecular_weight,charge,charge_density,isoelectric_point,gravy,instability_index,aromaticity,aliphatic_index,...,APAAC18,APAAC19,APAAC20,APAAC21,APAAC22,APAAC23,APAAC24,APAAC25,APAAC26,class
7191,MGQKTHPIGFRLGVIKEWPSKWYAPKKEYSKLLHEDLKIKNYIKER...,212,24155.25,21.404,0.000886,10.142029,-0.357547,26.849057,0.070755,100.660377,...,2.800,5.599,15.396,-0.001,-0.001,-0.000,0.000,-0.002,-0.001,0
7192,MSNALTNIFYKYVARRNSTWMAGAILGAFVLDSTVSGAVNTFFDSV...,66,7337.37,4.995,0.000681,9.898743,-0.022727,52.574394,0.136364,82.727273,...,3.288,4.932,11.509,0.001,0.001,-0.001,-0.002,0.000,-0.000,0
7193,MNDSVKTSLKRTLVGRVVSNKMDKTVTVLIEHRVKHPIYGKYVVRS...,90,10260.78,7.330,0.000714,9.808777,-0.492222,32.725556,0.044444,89.666667,...,1.384,4.153,19.384,-0.001,-0.000,-0.001,-0.001,-0.000,0.000,0
7194,ADGSDPASGEFLTEGGGVR,19,1821.86,-3.000,-0.001647,3.916565,-0.526316,34.300000,0.052632,46.315789,...,0.000,0.000,7.027,-0.001,-0.001,-0.001,-0.001,-0.000,0.000,0
7195,MGKPKFYDFCVHAVPDGDSTAQEQVSLGRHFGFSGIALANHSDRLP...,239,26213.90,-5.552,-0.000212,5.688049,-0.079079,42.247322,0.058577,101.213389,...,0.000,1.331,7.100,0.000,-0.000,-0.001,-0.000,-0.000,0.001,0


In [18]:
def get_embed_features(seq, pv2):
    features = {}
    embedding = pv2.to_vecs(seq)
    for i in range(3):
        for j in range(100):
            features["embed_{}_{}".format(i,j)] = embedding[i][j]
    return features

pv2 = prot_vec.load_protvec('swissprot-reviewed-protvec.model')
peptides_pos = datos_pos["sequence"].values
features_0 = get_embed_features('AAAAKAAALLLKLKKLLLKAAAAAAAAAAAAAAAA',pv2)
columns = list(features_0.keys())
df_embed_pos = pd.DataFrame(columns=columns)
for i in range(len(peptides_pos)):
    if i % 1000 == 0:
        print(i)
    features = get_embed_features(peptides_pos[i], pv2)
    df_embed_pos.loc[i] = [features[feature] for feature in features.keys()]

0
1000
2000
3000
4000
5000
6000
7000


In [19]:
peptides_neg = datos_neg["sequence"].values
features_0 = get_embed_features('AAAAKAAALLLKLKKLLLKAAAAAAAAAAAAAAAA',pv2)
columns = list(features_0.keys())
df_embed_neg = pd.DataFrame(columns=columns)
for i in range(len(peptides_neg)):
    if i % 1000 == 0:
        print(i)
    features = get_embed_features(peptides_neg[i], pv2)
    df_embed_neg.loc[i] = [features[feature] for feature in features.keys()]

0
1000
2000
3000
4000
5000
6000
7000


In [53]:
class_col = datos_pos["class"]
del datos_pos["class"]
datos_pos = pd.concat([datos_pos, df_embed_pos], axis=1)
datos_pos = pd.concat([datos_pos, class_col], axis=1)

In [54]:
datos_pos.tail()

,sequence,length,molecular_weight,charge,charge_density,isoelectric_point,gravy,instability_index,aromaticity,aliphatic_index,...,embed_2_91,embed_2_92,embed_2_93,embed_2_94,embed_2_95,embed_2_96,embed_2_97,embed_2_98,embed_2_99,class
7170,KPAWCWYTLAMCGAGYDSGTCDYMYSHCFGIKHHSSGSSSYHC,43,4800.35,0.050,0.000010,7.042786,-0.318605,52.651163,0.186047,25.116279,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,1
7171,DHYICVRSGGQCLYSACPIYTKIQGTCYHGKAKCCK,36,3999.68,3.785,0.000946,8.856262,-0.230556,47.391667,0.111111,56.944444,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,1
7172,VKSGHYKGPCYHDENCNGVCRDEGYKSGHCSRWGGACWCDT,41,4566.98,-0.112,-0.000025,6.948303,-1.043902,47.034146,0.121951,16.585366,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,1
7173,YEALVASILGKLSGLWHSDTVDFMGHTCHIRRRPKFRKFKLYHEGK...,95,10993.54,9.358,0.000851,9.952209,-0.573684,44.160000,0.126316,70.947368,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,1
7174,GGLKKLGKKLEGVGKRVFKASEKALPVLTGYKAI,34,3585.34,6.996,0.001951,10.297180,-0.152941,15.520588,0.058824,103.235294,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,1


In [55]:
class_col = datos_neg["class"]
del datos_neg["class"]
datos_neg = pd.concat([datos_neg, df_embed_neg], axis=1)
datos_neg = pd.concat([datos_neg, class_col], axis=1)

In [57]:
datos_neg.tail()

,sequence,length,molecular_weight,charge,charge_density,isoelectric_point,gravy,instability_index,aromaticity,aliphatic_index,...,embed_2_91,embed_2_92,embed_2_93,embed_2_94,embed_2_95,embed_2_96,embed_2_97,embed_2_98,embed_2_99,class
7191,MGQKTHPIGFRLGVIKEWPSKWYAPKKEYSKLLHEDLKIKNYIKER...,212,24155.25,21.404,0.000886,10.142029,-0.357547,26.849057,0.070755,100.660377,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,0
7192,MSNALTNIFYKYVARRNSTWMAGAILGAFVLDSTVSGAVNTFFDSV...,66,7337.37,4.995,0.000681,9.898743,-0.022727,52.574394,0.136364,82.727273,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,0
7193,MNDSVKTSLKRTLVGRVVSNKMDKTVTVLIEHRVKHPIYGKYVVRS...,90,10260.78,7.330,0.000714,9.808777,-0.492222,32.725556,0.044444,89.666667,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,0
7194,ADGSDPASGEFLTEGGGVR,19,1821.86,-3.000,-0.001647,3.916565,-0.526316,34.300000,0.052632,46.315789,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,0
7195,MGKPKFYDFCVHAVPDGDSTAQEQVSLGRHFGFSGIALANHSDRLP...,239,26213.90,-5.552,-0.000212,5.688049,-0.079079,42.247322,0.058577,101.213389,...,0.492228,-0.128685,0.093374,-0.083977,0.047865,-0.215438,0.019295,0.027602,0.034559,0


In [41]:
len(datos_pos.columns)

1810

In [58]:
datos_neg.to_csv("negativos_embed.csv")
datos_pos.to_csv("positivos_embed.csv")